In [0]:
# Databricks notebook source
from pyspark.sql.functions import avg, count, round, current_timestamp
import uuid

tabela_silver = "credit_risk.silver.silver_credit_risk"
tabela_gold = "credit_risk.gold.gold_loan_summary_by_purpose"
execution_id = str(uuid.uuid4())

try:
    df_silver = spark.read.table(tabela_silver)
    
    df_gold = df_silver.groupBy("purpose") \
        .agg(
            count("*").alias("total_emprestimos"),
            round(avg("credit_amount"), 2).alias("valor_medio_credito"),
            round(avg("duration"), 1).alias("prazo_medio_meses"),
            round(avg("age"), 1).alias("idade_media_solicitante")
        )

    df_gold.write.format("delta").mode("overwrite").saveAsTable(tabela_gold)
    
    spark.sql(f"""
        INSERT INTO credit_risk.bronze.log_pipeline_execution 
        VALUES ('{execution_id}', '03_gold_aggregation', '{tabela_gold}', {df_gold.count()}, 'SUCCESS', '', current_timestamp())
    """)
    print("Processamento Gold concluído.")

except Exception as e:
    erro = str(e).replace("'", "")
    spark.sql(f"""
        INSERT INTO credit_risk.bronze.log_pipeline_execution 
        VALUES ('{execution_id}', '03_gold_aggregation', '{tabela_gold}', 0, 'FAILED', '{erro}', current_timestamp())
    """)
    raise e